# Modélisation : Détection de Fraude par Random Forest
---
Dans cette étape, je construis mon modèle prédictif. J'ai choisi le **Random Forest** pour sa robustesse et sa capacité à gérer les poids de classes, ce qui est crucial ici.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, recall_score, classification_report
import joblib
import os

# Je charge les données
df = pd.read_csv('../data/creditcard.csv')

# Préparation : Je scale le temps et le montant car ce sont les seules variables non transformées
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time'] = scaler.fit_transform(df[['Time']])

X = df.drop('Class', axis=1)
y = df['Class']

# Je sépare en 80/20 avec stratify pour garder le ratio de fraude dans les deux sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Taille du set d\'entraînement : {len(X_train)}')
print(f'Taille du set de test : {len(X_test)}')

## 1. Entraînement avec pondération des classes
Je choisis `class_weight='balanced'` pour forcer mon modèle à accorder plus d'importance aux rares cas de fraude.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

## 2. Évaluation : Pourquoi AUC-PR ?
Sur des données très déséquilibrées, l'AUC-ROC peut être optimiste car elle prend en compte les Vrais Négatifs (qui sont ici massifs). J'utilise l'**AUC-PR** (Area Under the Precision-Recall Curve) car elle se concentre uniquement sur la performance par rapport à la classe positive (la fraude). C'est beaucoup plus exigeant et honnête.

In [ ]:
aucapr = average_precision_score(y_test, y_probs)
recall = recall_score(y_test, y_pred)

print(f'AUC-PR : {aucapr:.4f}')
print(f'Recall : {recall:.4f}')
print('\nClassification Report :\n', classification_report(y_test, y_pred))

## 3. Analyse SHAP
Je vais maintenant expliquer comment mon modèle prend ses décisions. C'est essentiel pour qu'un expert bancaire puisse justifier pourquoi une transaction a été bloquée.

In [ ]:
import shap

# J'utilise un échantillon pour SHAP car le dataset est grand
explainer = shap.TreeExplainer(model)
X_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(X_sample)

# Je gère la dimension des shap_values pour être sûr de l'affichage
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_sample, plot_type='dot')
else:
    shap.summary_plot(shap_values, X_sample, plot_type='dot')

## Conclusions Business
1. **Efficacité** : Mon modèle parvient à détecter une grande majorité des fraudes tout en limitant les faux positifs. C'est l'équilibre parfait pour ne pas bloquer les clients honnêtes.
2. **Transparence** : Grâce à SHAP, je peux identifier les signatures de la fraude. Certaines variables cachées (V1-V28) ont un impact massif, ce qui suggère des motifs de fraude sophistiqués.

In [ ]:
# Je sauvegarde le modèle pour mon application Streamlit
os.makedirs('../app', exist_ok=True)
joblib.dump(model, '../app/model.joblib')
joblib.dump(X.columns.tolist(), '../app/model_columns.joblib')
print('Modèle et colonnes sauvegardés.')